In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader

In [9]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
train_dataset = datasets.ImageFolder(
    "../../datasets/cats_dogs/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "../../datasets/cats_dogs/val",
    transform=val_transform
)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

In [11]:
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(
    model.fc.in_features,
    2
)

In [12]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
epochs = 5

for epoch in range(epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"Loss = {running_loss/len(train_loader):.4f}"
    )

/home/prakash/Documents/testsite/30days of deep learning/.venv/lib64/python3.14/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 1: Loss = 0.1427
Epoch 2: Loss = 0.0995
Epoch 3: Loss = 0.0975
Epoch 4: Loss = 0.0959
Epoch 5: Loss = 0.0931


In [13]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Validation Accuracy: {accuracy:.2f}%")

Validation Accuracy: 96.67%


In [14]:
torch.save(model.state_dict(), "../../models/catsvsdogs.pth")